In [18]:
# Install libraries
!pip install --upgrade pip setuptools wheel -q
!pip uninstall -y transformers datasets evaluate tokenizers accelerate rouge_score
!pip install tokenizers==0.14.1 -q
!pip install transformers==4.30.2 datasets evaluate rouge_score accelerate -q

import pandas as pd
import numpy as np
import torch
import evaluate
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    EarlyStoppingCallback
)
from datasets import Dataset, DatasetDict
from tqdm import tqdm

# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Found existing installation: tokenizers 0.22.2
Uninstalling tokenizers-0.22.2:
  Successfully uninstalled tokenizers-0.22.2
Found existing installation: accelerate 1.12.0
Uninstalling accelerate-1.12.0:
  Successfully uninstalled accelerate-1.12.0
Found existing installation: rouge_score 0.1.2
Uninstalling rouge_score-0.1.2:
  Successfully uninstalled rouge_score-0.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.0/295.0 kB 26.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchtune 0.6.1 requires datasets, which is not installed.
sentence-transformers 5.2.0 requires transformers<6.0.0,>=4.41.0, which is not installed.
peft 0.18.0 requires accelerate>=0.21.0, which is not installed.
peft 0.18.0 requires transformers, which is not installed.
gradio 5.50.0 requires

In [ ]:
# Load all training files
print("Loading Data...")
df1 = pd.read_csv("train.csv")
df2 = pd.read_csv("train_2.csv")
df3 = pd.read_csv("train_3.csv")

# Concatenate them into one big dataset
full_train_df = pd.concat([df1, df2, df3], ignore_index=True)

# Basic cleaning to prevent errors
def clean_text(text):
    return str(text).replace('\r\n', '\n').strip()

full_train_df['dialogue'] = full_train_df['dialogue'].apply(clean_text)
full_train_df['summary'] = full_train_df['summary'].apply(clean_text)

# Create a 5% validation split (Saving 95% for training)
train_val_split = full_train_df.sample(frac=1, random_state=42)
train_size = int(0.95 * len(full_train_df))

train_df = train_val_split[:train_size]
val_df = train_val_split[train_size:]

# Convert to Hugging Face format
raw_datasets = DatasetDict({
    "train": Dataset.from_pandas(train_df),
    "validation": Dataset.from_pandas(val_df)
})

print(f"Data Ready: {len(train_df)} Training | {len(val_df)} Validation")

📊 Loading Data...
✅ Data Ready: 1874 Training | 99 Validation


In [20]:
# Using your best performing model
MODEL_CHECKPOINT = "linydub/bart-large-samsum"
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

max_input_length = 1024
max_target_length = 128

def preprocess_function(examples):
    inputs = [str(d) for d in examples["dialogue"]]
    model_inputs = tokenizer(inputs, max_length=max_input_length, truncation=True)

    labels = tokenizer(text_target=[str(s) for s in examples["summary"]],
                       max_length=max_target_length, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = raw_datasets.map(preprocess_function, batched=True)
print("✅ Tokenization Complete")

Map:   0%|          | 0/1874 [00:00<?, ? examples/s]

Map:   0%|          | 0/99 [00:00<?, ? examples/s]

✅ Tokenization Complete


In [22]:
from transformers import IntervalStrategy
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_CHECKPOINT, use_safetensors=True)
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)
rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    # Replace -100 in the labels as we can't decode them
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    return {k: round(v * 100, 4) for k, v in result.items()}

# Robust hyperparameters
args = Seq2SeqTrainingArguments(
    "bart-samsum-final",
    learning_rate=2e-5,           # Low LR for fine-tuning stability
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    weight_decay=0.1,             # Higher weight decay to prevent overfitting
    label_smoothing_factor=0.1,   # Helps with generalization
    save_total_limit=1,
    num_train_epochs=3,           # 3 epochs is ideal for this dataset size
    eval_strategy=IntervalStrategy.EPOCH,
    save_strategy=IntervalStrategy.EPOCH,
    load_best_model_at_end=True,  # Always keeps the best version
    fp16=True,                    # Faster training on GPU
    report_to="none",
    predict_with_generate=True    # This will ensure predictions are token IDs
)

trainer = Seq2SeqTrainer(
    model,
    args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)

/tmp/ipython-input-2643018375.py:34: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [ ]:
print(" Starting Training...")
trainer.train()

# Check the final validation score
print("Evaluating on Validation Set...")
metrics = trainer.evaluate()
print(f"Final ROUGE-L Score on Validation: {metrics['eval_rougeL']}")

🚀 Starting Training...


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1,No log,2.483549,62.342400,42.004900,57.679100,57.507500
2,2.580200,2.366673,61.821500,41.127200,56.432600,56.325700
3,2.124100,2.372928,62.725100,43.070700,57.703200,57.578200


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 62, 'min_length': 11, 'early_stopping': True, 'num_beams': 6, 'no_repeat_ngram_size': 3}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


📊 Evaluating on Validation Set...


Final ROUGE-L Score on Validation: 56.4326


In [24]:
def polish_summary(text):
    if not isinstance(text, str): return ""
    text = text.strip()

    # Capitalize first letter
    if text and text[0].islower():
        text = text[0].upper() + text[1:]

    # Ensure punctuation at end
    if text and text[-1] not in ['.', '!', '?']:
        text = text + "."

    # Fix duplicate sentences (e.g., "Hello. Hello.")
    sentences = text.split('. ')
    if len(sentences) == 2 and sentences[0] == sentences[1].strip('.'):
        return sentences[0] + "."

    return text

In [ ]:
print("Generating Summaries for Test Set...")
test_df = pd.read_csv("test_2.csv")
model.to(device)
model.eval()

final_summaries = []
batch_size = 8
inputs = test_df["dialogue"].tolist()

# Loop through test data
for i in tqdm(range(0, len(inputs), batch_size)):
    batch = inputs[i:i+batch_size]
    model_inputs = tokenizer(batch, max_length=1024, padding=True, truncation=True, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **model_inputs,
            max_new_tokens=128,
            min_length=10,
            num_beams=5,             # 5 beams searches for the best path
            length_penalty=1.0,      # Neutral penalty
            no_repeat_ngram_size=3,  # Stops repetitive phrases
            early_stopping=True
        )

    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    # Apply the Polishing Technique
    polished = [polish_summary(s) for s in decoded]
    final_summaries.extend(polished)

# Save Submission
submission_df = pd.DataFrame({
    "id": test_df["id"],
    "summary": final_summaries
})

submission_df.to_csv("submission_winning_combo.csv", index=False)
print(" DONE! Download 'submission_winning_combo.csv'")

🚀 Generating Summaries for Test Set...


100%|██████████| 63/63 [00:57<00:00,  1.10it/s]

✅ DONE! Download 'submission_winning_combo.csv'
